In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
from itertools import product
from pathlib import Path

import pandas as pd
from strat import STRAT

from dema_strat import (
  compare_benchmark_summary,
  db_connect,
  global_style,
  grid_tuning,
  load_data_and_split,
)
from exits_builder import build_be_exits
from monte_carlo import parametic_simulation
from params_finder import neighborhood_analysis, radius_diagnostics
from sizer import sl_based_sizer
from visualizer import (
  combinations_compare_visualizer,
  portfolio_perf_visualizer,
  sensitivity_check_visualizer,
  trades_visualizer,
)

sys.path.insert(0, str(Path.cwd().parent))
from constants import CASH

# CONFIG
SPLIT_RATIO = 0.7
TABLE_NAME = 'f0'
TABLE_NAME_OOS = f"{TABLE_NAME}_OOS"
DATA_PATH = '~/market_data/XAUUSD/XAUUSD_M15.csv'
TARGET = 'profit_factor'
RISK_PER_TRADE = 10
RISK_PCT = CASH / RISK_PER_TRADE 

PARAMS = {
    "ema_period": {"type": "INTEGER", "data": range(5, 15), "default": 9},
    "smoothing_period": {"type": "INTEGER", "data": range(6, 16), "default": 12},
    "use_vwap": {"type": "BOOL", "data": [True, False], "default": True},
    "atr_period": {"type": "INTEGER", "data": range(14, 22, 2), "default": 18},
    "sl_mult": {"type": "REAL", "data": range(1,3), "default": 1},
    "rr_ratio": {"type": "REAL", "data": [0.25, 0.5, 1.0, 1.5, 2.0], "default": 1},
}

PARAM_COLS = list(PARAMS)
PARAMS_GRID = list(product(*(v["data"] for v in PARAMS.values())))
PARAMS_DEFAULT = {
    k: v.get("default", v["data"][0]) 
    for k, v in PARAMS.items()
}

TOP_N = max(100, len(PARAMS_GRID))

conn, cursor = db_connect()

def run_backtest(data, params = PARAMS_DEFAULT):
  # build the strat
  strat = STRAT(data, params)

  rr_ratio = params.get("rr_ratio", 2)
  sl_mult = params.get("sl_mult", 1)

  # exits and sizer
  long_exits, short_exits, sl_dist = build_be_exits(data, strat.signals, rr_ratio=rr_ratio, sl_mult=sl_mult)
  sizer = sl_based_sizer(risk_per_trade=RISK_PER_TRADE, sl_dists=sl_dist)

  portfolio = strat._run_backtest(long_exits, short_exits, size=sizer, size_type="amount")
  return portfolio, strat

df_ios, df_oos = load_data_and_split(DATA_PATH, split_ratio=SPLIT_RATIO)

## Grid tuning

In [ ]:
grid_tuning(params=PARAMS, table_name=TABLE_NAME, data=df_ios, run_backtest=run_backtest, replace_table=True)
grid_tuning(params=PARAMS, table_name=f"{TABLE_NAME}_OOS", data=df_oos, run_backtest=run_backtest, replace_table=True)

## Combinations compare for TOP 200 vs All combinations

In [ ]:
def get_query(limit = True):
    return f"""
        SELECT *
        FROM {TABLE_NAME}
        ORDER BY {TARGET} DESC
        {f'LIMIT {TOP_N}' if limit else ''}
    """

top_df = pd.read_sql_query(get_query(), conn)
all_df = pd.read_sql_query(get_query(limit = False), conn)

print(f"Top {TOP_N} combinations")
display(top_df)

combinations_compare_visualizer(
  first_data=top_df,
  second_data=all_df,
  chart_title=f"TOP {TOP_N} vs All combinations",
  first_name="TOP 200",
  second_name="All combinations"
)

## Combinations compare for OOS vs IOS

In [ ]:

def get_query(table):
    return f"""
        SELECT *
        FROM {table}
        ORDER BY {TARGET} DESC
        LIMIT {TOP_N}
    """

ios = pd.read_sql_query(get_query(TABLE_NAME), conn)
oos = pd.read_sql_query(get_query(f"{TABLE_NAME}_OOS"), conn)

print(f"OOS combinations (TOP {TOP_N})")
display(oos)

combinations_compare_visualizer(
  first_data=oos,
  second_data=ios,
  chart_title=f"OOS vs IOS (TOP {TOP_N})",
  first_name="OOS",
  second_name="IOS"
)

## Parameter neighborhood analysis 

In [ ]:
df = pd.read_sql_query(f"""
    SELECT *
    FROM {TABLE_NAME}
""", conn)

report, best_r = radius_diagnostics(df, PARAMS, TARGET, radii=(1, 2, 3, 4))
best_combination = neighborhood_analysis(df, PARAMS, TARGET=TARGET, radius=best_r)

## Sensitivity check

In [ ]:
df = pd.read_sql_query(f"""
    SELECT *
    FROM {TABLE_NAME}
""", conn)

sensitivity_check_visualizer(df, PARAM_COLS=PARAM_COLS, TARGET=TARGET)

## Benchmark

In [ ]:
best_combination_param = best_combination[PARAM_COLS].to_dict()
print(best_combination_param)

portfolio_ios, strat_ios = run_backtest(df_ios, params=best_combination_param)
portfolio_oos, strat_oos = run_backtest(df_oos, params=best_combination_param)

compare_benchmark_summary(
  data_ios=df_ios,
  data_oos=df_oos,
  portfolio_ios=portfolio_ios,
  portfolio_oos=portfolio_oos,
)

## Trade and Portfolio performance (IOS)

In [ ]:
trades_visualizer(data=df_ios, signals=strat_ios.signals, portfolio=portfolio_ios, N = 500)
portfolio_perf_visualizer(portfolio_ios)

## Trade and Portfolio performance (OOS)

In [ ]:
trades_visualizer(data=df_oos, signals=strat_oos.signals, portfolio=portfolio_oos, N = 500)
portfolio_perf_visualizer(portfolio_oos)

## Calc the EV for propfirm trading (TOPSTEPX and FTMO)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def solve_cost_R(win_rate_pct, profit_factor, rr_ratio=2.0, sl_R=1.0):
    """
    Back-solve the per-trade cost (in R units) from win rate, PF and RR.

    PF = p * (W - c) / ((1 - p) * (L + c))
    =>  c = (p*W - PF*(1-p)*L) / (PF*(1-p) + p)
    """
    p = win_rate_pct / 100
    W = rr_ratio * sl_R      # gross win in R
    L = sl_R                 # gross loss in R
    c = (p * W - profit_factor * (1 - p) * L) / (profit_factor * (1 - p) + p)
    return c


# ---- your strategy ----
win_rate = best_combination_param["winrate"]
pf       = best_combination_param["profit_factor"]
rr       = best_combination_param["rr_ratio"]

c = solve_cost_R(win_rate, pf, rr)

# ---- parameters ----
n_sims     = 10_000
n_trades   = 1_000         

p_win    = win_rate / 100
net_win  = rr - c
net_loss = -(1 + c)

risk_pct   = 0.01         
ruin_level = 1          

# ---- simulate ----
wins = rng.random((n_sims, n_trades)) < p_win
R = np.where(wins, net_win, net_loss)

# compounding equity
equity = CASH * np.cumprod(1 + risk_pct * R, axis=1)
equity = np.hstack([np.full((n_sims, 1), CASH), equity])

# ---- metrics ----
final = equity[:, -1]
peak = np.maximum.accumulate(equity, axis=1)
max_dd = ((peak - equity) / peak).max(axis=1)
ruined = (equity.min(axis=1) <= CASH * ruin_level)

print(f"Median final equity : {np.median(final):,.0f}")
print(f"5th / 95th pct      : {np.percentile(final,5):,.0f} / {np.percentile(final,95):,.0f}")
print(f"P(profit)           : {(final > CASH).mean():.1%}")
print(f"Median max DD       : {np.median(max_dd):.1%}")
print(f"95th pct max DD     : {np.percentile(max_dd,95):.1%}")
print(f"P(ruin @ {ruin_level:.0%})     : {ruined.mean():.1%}")

In [ ]:
best_combination_dict = best_combination.to_dict()

# ---- your strategy ----
win_rate = best_combination_dict["winrate"]
profit_factor = best_combination_dict["profit_factor"]
rr_ratio = best_combination_dict["rr_ratio"]

ftmo_equity = 1_000 * 30 # max drawdown * gold leverage
parametic_simulation(
  start_equity=ftmo_equity, 
  cost=100,
  payout=1000 + 100, # 10% + 100$ cost (FTMO refund)
  profit_target= 2500, # 15% of 2 step challenge + 10% funded
  win_rate=win_rate,
  profit_factor=profit_factor,
  # win_rate=60,
  # profit_factor=1.5,
  rr_ratio=rr_ratio,
  risk_pct=0.01,
  verbose=True
)

In [ ]:
rng = np.random.default_rng(42)

trades = portfolio_ios.trades.records_readable

# ---- parameters ----
n_sims     = 10_000
n_trades   = 1_000         

p_win      = best_combination_param["winrate"] / 100

risk_pct   = 0.01         
ruin_level = 1          

# ---- simulate ----
trades_R = trades["PnL"] / RISK_PER_TRADE
trades_R = trades_R.values

idx = rng.integers(0, len(trades_R), size=(n_sims, n_trades))
R = trades_R[idx]


# compounding equity
equity = CASH * np.cumprod(1 + risk_pct * R, axis=1)
equity = np.hstack([np.full((n_sims, 1), CASH), equity])

# ---- metrics ----
final = equity[:, -1]
peak = np.maximum.accumulate(equity, axis=1)
max_dd = ((peak - equity) / peak).max(axis=1)
ruined = (equity.min(axis=1) <= CASH * ruin_level)

print(f"Median final equity : {np.median(final):,.0f}")
print(f"5th / 95th pct      : {np.percentile(final,5):,.0f} / {np.percentile(final,95):,.0f}")
print(f"P(profit)           : {(final > CASH).mean():.1%}")
print(f"Median max DD       : {np.median(max_dd):.1%}")
print(f"95th pct max DD     : {np.percentile(max_dd,95):.1%}")
print(f"P(ruin @ {ruin_level:.0%})     : {ruined.mean():.1%}")